# Microbiology Workflow Analysis – TAT & Delay Investigation

## Context
This analysis simulates an Epic Beaker-aligned laboratory workflow, focusing on the order-to-result lifecycle across microbiology and molecular diagnostics.

The dataset includes:
- Location-based workflow variation (ICU, ED, FSED, PrimaryCare)
- Priority-based differences (STAT vs Routine)
- Controlled operational variability (transport delays, batching, instrument queue)

## Objective
Identify where delays occur across the workflow:
- Collection
- Transport
- Lab processing
- Result posting

## Why this matters
In real clinical systems, delays impact:
- Patient care turnaround time
- ED throughput
- Lab efficiency and staffing

In [ ]:
from microbiology_workflow_analysis_generator import (
    build_orders,
    add_workflow_metrics,
    add_event_flags,
)

df = build_orders()
df = add_workflow_metrics(df)
df = add_event_flags(df)

df.head()

## Dataset Overview

The dataset simulates microbiology and molecular test orders across inpatient, emergency, off-site, and outpatient-style locations.

Each row represents one laboratory order moving through the order-to-result lifecycle:

`order_time → collect_time → receive_time → instrument_result_time → final_result_time`

The redone generator also includes operational exception logic through the `delay_reason` field. This makes the dataset more realistic because not every delay comes from the same workflow stage.


In [ ]:
print("Rows, Columns:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nPatient Location Distribution:")
print(df["patient_location"].value_counts())

print("\nPriority Distribution:")
print(df["priority"].value_counts())

print("\nWorkflow Type Distribution:")
print(df["workflow_type"].value_counts())

print("\nDelay Reason Distribution:")
print(df["delay_reason"].value_counts())

## Overall Turnaround Time Profile

The first step is to establish the overall TAT profile across collection, transport, lab processing, downstream result posting, and total turnaround time.

This helps separate normal clinical workflow duration from true operational delay.


In [ ]:
df[[
    "tat_collection_min",
    "tat_transport_min",
    "tat_lab_min",
    "tat_result_post_min",
    "tat_total_min",
    "tat_total_hours",
]].describe().round(2)

## Turnaround Time by Patient Location

Location is one of the strongest workflow drivers in laboratory operations.

Inpatient and ED workflows usually have shorter collection and transport paths, while FSED and PrimaryCare workflows are more exposed to courier batching, transport lag, and receipt delays.


In [ ]:
df.groupby("patient_location")["tat_total_hours"] \
    .mean() \
    .sort_values(ascending=False) \
    .round(2)

In [ ]:
df.groupby("patient_location")[[
    "tat_collection_min",
    "tat_transport_min",
    "tat_lab_min",
    "tat_result_post_min",
]].mean().round(1)

### Analyst Interpretation

This view should be read as a stage-segmentation table, not just a TAT table.

If total TAT is high, the analyst needs to determine whether the root driver is:

- Collection delay before specimen acquisition
- Transport / receipt delay before the lab has the specimen
- Lab processing delay after receipt
- Downstream posting delay after instrument/result completion

This is the same mental model used when troubleshooting order-to-result issues in LIS or Epic Beaker workflows.


## Delay Reason Analysis

The redone generator adds `delay_reason`, which turns this from a simple TAT exercise into a troubleshooting-style workflow investigation.

This field allows the analyst to ask: *What operational exception most often explains delayed orders?*


In [ ]:
df["delay_reason"].value_counts()

In [ ]:
df[df["delay_reason"] != "None"] \
    .groupby(["delay_reason", "patient_location", "priority"]) \
    .size() \
    .reset_index(name="order_count") \
    .sort_values("order_count", ascending=False) \
    .head(20)

### Analyst Interpretation

This is now much closer to real-world clinical systems work.

Instead of only saying “TAT is high,” the analysis can identify whether delays are associated with:

- Courier batching
- ED/ICU STAT exceptions
- Specimen collection delays
- Instrument queue or manual handling
- Downstream result posting issues

That makes the dataset better for SQL practice because you can group, filter, and summarize by both workflow stage and operational cause.


## STAT vs Routine Performance

Priority handling is critical in clinical operations.

STAT workflows should generally perform faster than routine workflows. However, the updated generator intentionally allows a subset of STAT orders to fail expected thresholds, especially through rare ED/ICU operational exceptions or high lab-processing duration for culture-based tests.


In [ ]:
df.groupby("priority")["tat_total_hours"].mean().round(2)

In [ ]:
df.groupby("priority")[[
    "tat_collection_min",
    "tat_transport_min",
    "tat_lab_min",
    "tat_result_post_min",
]].mean().round(1)

In [ ]:
df["stat_delay_flag"].value_counts()

In [ ]:
df[df["stat_delay_flag"] == "Y"][[
    "order_id",
    "patient_location",
    "priority",
    "test_name",
    "workflow_type",
    "delay_reason",
    "tat_total_hours",
    "tat_collection_min",
    "tat_transport_min",
    "tat_lab_min",
    "tat_result_post_min",
]].sort_values("tat_total_hours", ascending=False).head(15)

### STAT Delay Interpretation

For STAT delays, avoid assuming the problem is always transport.

A STAT delay can be caused by:

- Transport delay from ED/ICU exception events
- Lab processing time for cultures
- Instrument queue or manual handling
- Result posting delay

This is why segmented TAT is more useful than total TAT alone.


## Transport Delay Investigation

The updated generator uses context-aware transport thresholds.

This matters because a flat threshold can hide ED/ICU STAT delays. For example, a 55-minute transport time may be unacceptable for ED STAT work, while a longer transport time may be more expected for off-site courier workflows.


In [ ]:
df.groupby(["patient_location", "priority"])["transport_delay_flag"] \
    .value_counts() \
    .unstack(fill_value=0)

In [ ]:
transport_summary = (
    df.groupby(["patient_location", "priority"])
    .agg(
        orders=("order_id", "count"),
        avg_transport_min=("tat_transport_min", "mean"),
        delayed_transport_orders=("transport_delay_flag", lambda x: (x == "Y").sum()),
    )
    .reset_index()
)

transport_summary["transport_delay_rate_pct"] = (
    transport_summary["delayed_transport_orders"] / transport_summary["orders"] * 100
).round(1)

transport_summary.sort_values(
    ["transport_delay_rate_pct", "avg_transport_min"],
    ascending=False
)

In [ ]:
df[df["transport_delay_flag"] == "Y"][[
    "order_id",
    "patient_location",
    "priority",
    "test_name",
    "delay_reason",
    "tat_transport_min",
    "tat_total_hours",
]].sort_values("tat_transport_min", ascending=False).head(20)

### Transport Delay Interpretation

This section is now stronger than the original version because it distinguishes between:

- High-volume off-site transport risk
- Context-sensitive ED/ICU STAT transport failures
- Routine delay patterns that are expected but still operationally relevant

This is the kind of logic that could support a Clarity-style dashboard or a Beaker validation review.


## Bottleneck Flag Analysis

Threshold-based event flags simulate operational alert logic that might appear in reporting layers, workqueues, dashboards, or validation reports.

The flags help classify which workflow stage is most likely contributing to delay.


In [ ]:
flag_cols = [
    "collection_delay_flag",
    "transport_delay_flag",
    "lab_delay_flag",
    "result_post_delay_flag",
    "stat_delay_flag",
]

for col in flag_cols:
    print(f"\n{col}")
    print(df[col].value_counts())

In [ ]:
flag_summary = []

for col in flag_cols:
    flag_summary.append({
        "flag": col,
        "flagged_orders": (df[col] == "Y").sum(),
        "flag_rate_pct": round((df[col] == "Y").mean() * 100, 1),
    })

import pandas as pd

pd.DataFrame(flag_summary).sort_values("flag_rate_pct", ascending=False)

## Workflow Type Comparison

Molecular and culture workflows have fundamentally different processing characteristics.

Molecular tests are usually measured in hours. Culture workflows are measured in longer windows because growth, incubation, workup, and finalization are part of the clinical process.


In [ ]:
df.groupby("workflow_type")["tat_total_hours"].mean().round(2)

In [ ]:
df.groupby("test_name")["tat_total_hours"] \
    .mean() \
    .sort_values(ascending=False) \
    .round(2)

In [ ]:
df.groupby(["workflow_type", "priority"])[[
    "tat_collection_min",
    "tat_transport_min",
    "tat_lab_min",
    "tat_result_post_min",
    "tat_total_hours",
]].mean().round(2)

## SQL Translation Targets

The strongest SQL investigations from this dataset are:

1. Transport delay rates by location and priority
2. STAT delay review with segmented TAT
3. Delay reason counts by location
4. Average stage-level TAT by workflow type
5. Top delayed orders for operational review
6. Exception-rate comparison between ED/ICU and off-site workflows

These are realistic analyst questions because they move from raw timestamps to operational interpretation.


## SQL Translation Example (Clarity-style)

```sql
SELECT
    patient_location,
    priority,
    delay_reason,
    COUNT(*) AS delayed_orders
FROM microbiology_workflow
WHERE transport_delay_flag = 'Y'
GROUP BY patient_location, priority, delay_reason
ORDER BY delayed_orders DESC;

# Executive Summary

- Transport delays are the primary driver of extended TAT, especially in FSED and PrimaryCare workflows
- STAT workflows perform better overall, but ED/ICU exceptions still occur due to operational variability
- Culture workflows predictably exceed molecular TAT due to incubation requirements
- Delay segmentation enables targeted workflow optimization rather than global fixes

# Recommended Next Steps

- Implement transport monitoring alerts for outreach locations
- Develop STAT exception dashboards for ED/ICU workflows
- Separate workflow stage metrics in reporting (collection vs transport vs lab)
- Extend analysis into SQL-based Clarity reporting for real-time monitoring
